# Third-Party Cross-Sell Decision Data — Ingestion

Parses raw Pega decision JSON exports, extracts all third-party cross-sell variant
(BookingDotCom / Cartrawler) scoring events for the
**Email / Outbound / ThirdParty / Sales** partition, and writes a clean Parquet file
to `data/processed/`.

**To add new data:** drop additional JSON export files into `data/decisions/` and re-run.
Deduplication on `modelExecutionID` ensures records are never double-counted.

In [1]:
from pathlib import Path
import json
from collections import Counter

import pandas as pd
import polars as pl
from IPython.display import HTML, display
from tabulate import tabulate
from tqdm import tqdm

from my_project.parsing import extract_l5b15_rows, THIRDPARTY_VARIANTS

print(f"Imports OK — targeting: {sorted(THIRDPARTY_VARIANTS)}")

Imports OK — targeting: ['BookingDotCom', 'Cartrawler']


In [2]:
DECISIONS_DIR = Path("../data/decisions")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = PROCESSED_DIR / "thirdparty_email_outbound.parquet"

# All decision JSON exports — add new files to data/decisions/ and re-run
JSON_FILES = sorted(DECISIONS_DIR.glob("*.json"))
print(f"Found {len(JSON_FILES)} source file(s):")
for f in JSON_FILES:
    print(f"  {f.name}")

Found 3 source file(s):
  data.json
  data_2.json
  data_3.json


## Load & parse raw records

In [3]:
CHUNK_SIZE = 5_000  # records per chunk — keeps peak RAM well under 1 GB per file

all_rows = []

for filepath in JSON_FILES:
    print(f"\nProcessing {filepath.name}...")
    with open(filepath, encoding="utf-8") as f:
        total = sum(1 for _ in f)

    file_rows = 0
    chunk: list[dict] = []

    with open(filepath, encoding="utf-8") as f:
        for line in tqdm(f, total=total, desc=filepath.name):
            chunk.append(json.loads(line))
            if len(chunk) >= CHUNK_SIZE:
                extracted = extract_l5b15_rows(
                    pl.DataFrame(chunk),
                    target_names=list(THIRDPARTY_VARIANTS),
                )
                all_rows.extend(extracted)
                file_rows += len(extracted)
                chunk = []  # release memory before next chunk

    if chunk:  # process final partial chunk
        extracted = extract_l5b15_rows(
            pl.DataFrame(chunk),
            target_names=list(THIRDPARTY_VARIANTS),
        )
        all_rows.extend(extracted)
        file_rows += len(extracted)

    print(f"  {total:,} raw interactions  ->  {file_rows:,} third-party rows extracted")

print(f"\nTotal rows before deduplication: {len(all_rows):,}")


Processing data.json...


data.json: 100%|██████████| 16216/16216 [00:08<00:00, 1872.09it/s]


  16,216 raw interactions  ->  14,233 third-party rows extracted

Processing data_2.json...


data_2.json: 100%|██████████| 28564/28564 [00:15<00:00, 1874.97it/s]


  28,564 raw interactions  ->  29,609 third-party rows extracted

Processing data_3.json...


data_3.json: 100%|██████████| 53047/53047 [00:30<00:00, 1762.21it/s]


  53,047 raw interactions  ->  56,941 third-party rows extracted

Total rows before deduplication: 100,783


## Deduplicate & filter

In [4]:
df_all = pd.DataFrame(all_rows)

# Deduplicate on modelExecutionID (safe to re-run with overlapping exports)
before = len(df_all)
df_all = df_all.drop_duplicates(subset=["modelExecutionID"])
print(f"After dedup: {len(df_all):,} rows  (removed {before - len(df_all):,} duplicates)")

# Filter to the channel/partition of interest
df = df_all[
    (df_all["pyChannel"]   == "Email") &
    (df_all["pyDirection"] == "Outbound") &
    (df_all["pyGroup"]     == "ThirdParty") &
    (df_all["pyIssue"]     == "Sales")
].copy().reset_index(drop=True)

# Parse interaction timestamp
df["pxDecisionTime"] = pd.to_datetime(
    df["pxDecisionTime"], format="%Y%m%dT%H%M%S.%f %Z", utc=True, errors="coerce"
)

print(f"After filter (Email / Outbound / ThirdParty / Sales): {len(df):,} rows")
print(f"Columns: {df.shape[1]}")
df.head(3)

After dedup: 100,725 rows  (removed 58 duplicates)
After filter (Email / Outbound / ThirdParty / Sales): 82,721 rows
Columns: 441


,pxInteractionID,pxSubjectID,modelExecutionID,modelVersion,pxDecisionTime,pyName,pyChannel,pyDirection,pyGroup,pyIssue,...,CustBookedFlight.PassengerBookingData.FlightSeason,CustBookedFlight.PassengerBookingData.ProductClass,CustBookedFlight.PassengerBookingData.DestinationType,CustBookedFlight.PassengerBookingData.BookingChannel,IH.Push.Outbound.Clicked.pyHistoricalOutcomeCount,IH.Push.Outbound.Clicked.pxLastGroupID,IH.Push.Outbound.Clicked.pxLastOutcomeTime.DaysSince,IH.Mobile.Inbound.Clicked.pyHistoricalOutcomeCount,IH.Mobile.Inbound.Clicked.pxLastGroupID,IH.Mobile.Inbound.Clicked.pxLastOutcomeTime.DaysSince
0,-3097192767374793999,M-6843641,63172f4d-363e-4919-883e-79cbd89ef9f6,c0a39e52-0934-5d41-b370-24347e325675,2026-04-04 09:41:36.661000+00:00,Cartrawler,Email,Outbound,ThirdParty,Sales,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,-3097192767374793999,M-6843641,eb2814cc-4e17-479b-b51d-cb89a1497899,f07d34e6-f844-5682-bfd5-b1aeea3bb89e,2026-04-04 09:41:36.661000+00:00,Cartrawler,Email,Outbound,ThirdParty,Sales,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,-3097192767374793999,M-6843641,51468987-b275-4ac7-99d6-c709a7eb7ba4,0972ccac-7b6c-5785-bfbf-e289fe10a7cf,2026-04-04 09:41:36.661000+00:00,BookingDotCom,Email,Outbound,ThirdParty,Sales,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Partition & model version overview

In [5]:
# Product variant breakdown
print("Decisions per third-party variant:")
print(df["pyName"].value_counts(dropna=False).to_string())

print()
dt = df["pxDecisionTime"].dropna()
if len(dt):
    print(f"Interaction date range: {dt.min().date()} -> {dt.max().date()}")

Decisions per third-party variant:
pyName
Cartrawler       42160
BookingDotCom    40561

Interaction date range: 2026-03-09 -> 2026-04-30


In [6]:
# Propensity by variant
print("Propensity summary by pyName:")
print(df.groupby("pyName")["propensity"].describe().round(4).to_string())

Propensity summary by pyName:
                 count    mean     std     min     25%     50%     75%     max
pyName                                                                        
BookingDotCom  40561.0  0.0006  0.0003  0.0002  0.0004  0.0005  0.0007  0.0096
Cartrawler     42160.0  0.0021  0.0012  0.0005  0.0018  0.0019  0.0022  0.0326


## Save to Parquet

In [7]:
df.to_parquet(OUTPUT_FILE, index=False)
print(f"Saved {len(df):,} rows -> {OUTPUT_FILE}")
print(f"File size: {OUTPUT_FILE.stat().st_size / 1024:.1f} KB")

Saved 82,721 rows -> ../data/processed/thirdparty_email_outbound.parquet
File size: 14960.5 KB
